Install library

In [5]:
!pip install plotly --quiet
print("Done!")

Done!


Import all libraries

In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
print("All libraries ready!")

All libraries ready!


Load and preview your data

In [7]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"]       = df["Order Date"].dt.year
df["Month"]      = df["Order Date"].dt.month
df["MonthName"]  = df["Order Date"].dt.strftime("%b")

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head(5)

Rows: 9994 | Columns: 24


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,Month,MonthName
0,1,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,Nov
1,2,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,Nov
2,3,CA-2016-138688,2016-06-12,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,Jun
3,4,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,Oct
4,5,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,Oct


KPI Summary

In [8]:
total_revenue = df["Sales"].sum()
total_orders  = df["Order ID"].nunique()
total_profit  = df["Profit"].sum()
gross_margin  = (total_profit / total_revenue) * 100
avg_order_val = df.groupby("Order ID")["Sales"].sum().mean()
total_customers = df["Customer ID"].nunique()

print("=" * 40)
print(f"  💰 Total Revenue    : ${total_revenue:,.2f}")
print(f"  🛒 Total Orders     : {total_orders:,}")
print(f"  👥 Total Customers  : {total_customers:,}")
print(f"  📈 Total Profit     : ${total_profit:,.2f}")
print(f"  📊 Gross Margin     : {gross_margin:.1f}%")
print(f"  🧾 Avg Order Value  : ${avg_order_val:.2f}")
print("=" * 40)

  💰 Total Revenue    : $2,297,200.86
  🛒 Total Orders     : 5,009
  👥 Total Customers  : 793
  📈 Total Profit     : $286,397.02
  📊 Gross Margin     : 12.5%
  🧾 Avg Order Value  : $458.61


Monthly Revenue Trend (bar + line chart)

In [9]:
monthly = (df.groupby(["Year","Month","MonthName"])["Sales"]
             .sum().reset_index().sort_values(["Year","Month"]))

fig = px.bar(monthly, x="MonthName", y="Sales", color="Year",
             barmode="group",
             title="📅 Monthly Revenue Trend by Year",
             color_discrete_sequence=["#378ADD","#1D9E75","#EF9F27","#D85A30"],
             labels={"Sales":"Revenue ($)","MonthName":"Month"})
fig.update_layout(plot_bgcolor="white", title_font_size=18,
                  legend_title="Year")
fig.show()

Revenue by Category (donut chart)

In [10]:
cat_rev = df.groupby("Category")["Sales"].sum().reset_index()

fig = px.pie(cat_rev, values="Sales", names="Category",
             title="🍩 Revenue Split by Category",
             color_discrete_sequence=["#378ADD","#1D9E75","#EF9F27"],
             hole=0.5)
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.update_layout(title_font_size=18)
fig.show()

Top 10 Products by Revenue

In [11]:
top10 = (df.groupby("Product Name")["Sales"]
           .sum().nlargest(10).reset_index()
           .sort_values("Sales"))

fig = px.bar(top10, x="Sales", y="Product Name",
             orientation="h",
             title="🏆 Top 10 Products by Revenue",
             color="Sales",
             color_continuous_scale="Blues",
             labels={"Sales":"Revenue ($)","Product Name":""})
fig.update_layout(plot_bgcolor="white", title_font_size=18,
                  coloraxis_showscale=False)
fig.show()

Sales & Profit by Region

In [12]:
region = df.groupby("Region")[["Sales","Profit"]].sum().reset_index()

fig = px.bar(region, x="Region", y=["Sales","Profit"],
             barmode="group",
             title="🗺️ Sales & Profit by Region",
             color_discrete_sequence=["#378ADD","#1D9E75"],
             labels={"value":"Amount ($)","variable":"Metric"})
fig.update_layout(plot_bgcolor="white", title_font_size=18)
fig.show()

Profit vs Sales Scatter (shows which products are actually profitable)

In [13]:
fig = px.scatter(df, x="Sales", y="Profit",
                 color="Category", size="Quantity",
                 hover_data=["Product Name","Region"],
                 title="💡 Profit vs Sales — by Category",
                 color_discrete_sequence=["#378ADD","#1D9E75","#EF9F27"])
fig.add_hline(y=0, line_dash="dash", line_color="red",
              annotation_text="Break-even line")
fig.update_layout(plot_bgcolor="white", title_font_size=18)
fig.show()

Yearly Revenue vs Profit (final summary chart)

In [14]:
yearly = df.groupby("Year")[["Sales","Profit"]].sum().reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=yearly["Year"], y=yearly["Sales"],
                     name="Revenue", marker_color="#378ADD"), secondary_y=False)
fig.add_trace(go.Scatter(x=yearly["Year"], y=yearly["Profit"],
                         name="Profit", mode="lines+markers",
                         line=dict(color="#1D9E75", width=3),
                         marker=dict(size=10)), secondary_y=True)
fig.update_layout(title="📊 Yearly Revenue vs Profit Trend",
                  plot_bgcolor="white", title_font_size=18)
fig.update_yaxes(title_text="Revenue ($)", secondary_y=False)
fig.update_yaxes(title_text="Profit ($)", secondary_y=True)
fig.show()